# 📚 Document-Combining Chains → LCEL

## Learning Objectives
In this notebook, you will learn:
1. **What `load_summarize_chain` actually built** - the three strategies hiding behind one `chain_type=` string
2. **Stuff, written out** - why the simplest strategy needs no chain class at all
3. **Map-reduce, written out** - and why the map half gets concurrency for free with `.batch()`
4. **Refine, written out** - the one strategy that genuinely cannot be parallelized
5. **Reading the silence** - why this API is easy to keep using long after it was retired

## Prerequisites
- `langchain >= 1.4.0`, `langchain-core >= 1.6.1`, `langchain-classic >= 1.0.8`,
  and `langchain-openai` installed (the floors in this repo's `pyproject.toml`)
- `OPENAI_API_KEY` in your project `.env`
- Notebooks `8.0_Summarization_Essentials` and `8.1_Text_Summarization`
- Comfort with LCEL piping (`3.1_LCEL_Introduction`)

---
## 🤔 Part 1: Why this changed

`load_summarize_chain(llm, chain_type="stuff" | "map_reduce" | "refine")` was a
convenient front door. You picked a string, got back a chain, called `.run(docs)`.

The problem is what the convenience cost you. The strategy — the actual interesting
part, the thing you would want to tune — was hidden inside a class you did not
write and could not easily inspect. Want a different reduce prompt? A retry on one
chunk only? To log which chunk was slowest? You were reading LangChain's source.

1.x's answer is that these three "strategies" are not framework features at all.
They are three short pieces of ordinary Python:

| Strategy | What it really is |
| --- | --- |
| `stuff` | join the documents, fill one prompt, call once |
| `map_reduce` | call once per chunk, then call once more on the results |
| `refine` | call once, then revise that answer once per remaining chunk |

Written out, each is 3-8 lines and every knob is in your hands.

### Key Concepts:
- **Combine strategy**: how you get N documents through a model with a finite context window
- **`langchain-classic`**: where the retired chain machinery lives now — installable, still runnable, not recommended

---
## ⚙️ Part 2: Setup

Direct provider instantiation, matching the other notebooks in
`LangChain_Fundamentals/` (the LangGraph-side phases use the `helpers` factory
instead — that convention is per-phase).

In [ ]:
# ==============================================================================
# ENVIRONMENT SETUP: model, sample documents
# ==============================================================================
import os

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# llm = ChatGroq(model="openai/gpt-oss-120b")
# llm = ChatAnthropic(model="claude-sonnet-4-5")

# Small inline corpus so this notebook needs no PDF and no network beyond the
# model call itself.
docs = [
    Document(page_content="LangChain 1.0 narrowed the top-level package to the "
                          "agent loop and a few convenience re-exports."),
    Document(page_content="Everything retired - chains, retrievers, the indexing "
                          "API, the prompt hub - moved to langchain-classic."),
    Document(page_content="LCEL was left untouched: prompt | llm | parser is "
                          "still the supported way to compose."),
]

print(f"🤖 Model loaded: {llm.model_name}")
print(f"📋 {len(docs)} sample document(s)")
print("✅ Setup complete!")

---
## 🕰️ Part 3: The old way

> ⚠️ Does not run on LangChain 1.x — shown for contrast only.

In [ ]:
# ==============================================================================
# LEGACY 0.X: ONE LOADER, THREE STRATEGIES
# ==============================================================================
from langchain.chains.summarize import load_summarize_chain

stuff_chain  = load_summarize_chain(llm, chain_type="stuff")
mapred_chain = load_summarize_chain(llm, chain_type="map_reduce")
refine_chain = load_summarize_chain(llm, chain_type="refine")

summary = stuff_chain.run(docs)

**Two separate problems with that snippet on 1.x**, and it is worth keeping them apart:

1. `langchain.chains` no longer exists — that is the package split, and the import
   raises `ModuleNotFoundError`. Repointing to `langchain_classic.chains.summarize`
   fixes *that*.
2. Repointing does not make the code current. What comes back is still a retired
   class, and `.run()` is still a retired call style.

---
## 🔍 Part 4: The silence is the interesting part

Here is the detail worth internalising. Construct the legacy chain and watch how
much LangChain tells you:

In [ ]:
# ==============================================================================
# DEPRECATION AUDIT: what the loader does and does not warn about
# ==============================================================================
import warnings

from langchain_classic.chains.summarize import load_summarize_chain

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    legacy_chain = load_summarize_chain(llm, chain_type="stuff")

print(f"📋 returned type      : {type(legacy_chain).__name__}")
print(f"⚠️  warnings at build  : {len(caught)}")

# ...and yet the class it handed back carries a deprecation marker. Read it off
# the class rather than taking this notebook's word for it:
print("📋 deprecation marker :", type(legacy_chain).__deprecated__)

`load_summarize_chain` builds a `StuffDocumentsChain`. Its source carries
`@deprecated(since="0.2.13", removal="2.0.0", alternative="langchain.agents.create_agent")`,
and the marker printed above is the sentence the decorator composes from its
`alternative=` argument — yet
constructing the chain through the loader emits **no warning at all**. You only hear from LangChain when you *call*
the chain, and then the warning is about `.run()`, not about the strategy class.

That is why this API is still everywhere: nothing tells you to stop.

---
## 📦 Part 5: `stuff`, written out

Join the documents, fill one prompt, call once. That is the whole strategy.

In [ ]:
# ==============================================================================
# STUFF: ONE PROMPT, ONE CALL
# ==============================================================================
stuff_prompt = ChatPromptTemplate.from_messages(
    [("human", "Write a concise summary of the following:\n\n{text}")]
)

stuff_chain = stuff_prompt | llm | StrOutputParser()

result = stuff_chain.invoke({"text": "\n\n".join(d.page_content for d in docs)})

print("🔧 stuff summary:")
print(result)

Use it when the whole corpus comfortably fits the context window. It is one API
call, so it is the cheapest and the least lossy — nothing gets summarized twice.

---
## 🗺️ Part 6: `map_reduce`, written out

Two chains and a `.batch()`. The map step is embarrassingly parallel, and writing
it out is what lets you *see* that.

In [ ]:
# ==============================================================================
# MAP-REDUCE: SUMMARIZE EACH, THEN SUMMARIZE THE SUMMARIES
# ==============================================================================
map_prompt = ChatPromptTemplate.from_messages(
    [("human", "Summarize this passage in one sentence:\n\n{text}")]
)
reduce_prompt = ChatPromptTemplate.from_messages(
    [("human", "Combine these summaries into one short paragraph:\n\n{text}")]
)

map_chain = map_prompt | llm | StrOutputParser()
reduce_chain = reduce_prompt | llm | StrOutputParser()

# --- MAP: one call per document, run concurrently by .batch() ---
partials = map_chain.batch([{"text": d.page_content} for d in docs])
print(f"🔧 mapped {len(partials)} document(s)")

# --- REDUCE: one more call over the joined partials ---
result = reduce_chain.invoke({"text": "\n\n".join(partials)})

print("🔧 map-reduce summary:")
print(result)

`.batch()` dispatches the map calls through a thread pool rather than looping.
That matters at 50 chunks and is invisible at one — with a single input `.batch()`
short-circuits and runs inline, so you will only see the benefit on a real corpus.

---
## 🔁 Part 7: `refine`, written out

A running summary, revised once per document. Sequential by construction: each
step needs the previous answer, so there is nothing to parallelize.

In [ ]:
# ==============================================================================
# REFINE: A RUNNING SUMMARY, REVISED ONE DOCUMENT AT A TIME
# ==============================================================================
first_prompt = ChatPromptTemplate.from_messages(
    [("human", "Write a concise summary of the following:\n\n{text}")]
)
refine_prompt = ChatPromptTemplate.from_messages(
    [("human",
      "Here is a running summary:\n{existing}\n\n"
      "Refine it using this additional context:\n{text}\n\n"
      "If the context adds nothing, return the summary unchanged.")]
)

summary = (first_prompt | llm | StrOutputParser()).invoke(
    {"text": docs[0].page_content}
)

refine_step = refine_prompt | llm | StrOutputParser()
for doc in docs[1:]:
    summary = refine_step.invoke({"existing": summary, "text": doc.page_content})

print(f"🔧 refined over {len(docs)} document(s)")
print(summary)

> **Note**: refine costs N sequential calls and its quality depends on document
> order — later documents get more influence. Map-reduce treats all chunks
> symmetrically and finishes in roughly the time of the slowest one. Refine wins
> when later context genuinely should override earlier context.

---
## 🔀 Part 8: Side by side

| 0.x | 1.x | Calls | Parallel? |
| --- | --- | --- | --- |
| `load_summarize_chain(llm, chain_type="stuff")` | `prompt \| llm \| StrOutputParser()` | 1 | n/a |
| `load_summarize_chain(llm, chain_type="map_reduce")` | `map_chain.batch(...)` then `reduce_chain.invoke(...)` | N+1 | map step yes |
| `load_summarize_chain(llm, chain_type="refine")` | one `.invoke()`, then a loop | N | no, by design |
| `chain.run(docs)` | `chain.invoke({"text": ...})` | — | — |

If you must keep the loader — for instance while porting a large codebase in
stages — it still exists at `langchain_classic.chains.summarize.load_summarize_chain`.
Treat that as a staging post, not a destination.

---
## 🚨 Part 9: Common errors when migrating

**1. The import**

```
ModuleNotFoundError: No module named 'langchain.chains'
```

The package split. `langchain_classic.chains.summarize` — or, better, stop using
the loader.

**2. Calling the chain the old way**

```
The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
```

A warning, not an error — the code still runs today. It will not in 2.0.

**3. Passing documents where the prompt wants a string**

`load_summarize_chain` accepted a list of `Document` objects. An LCEL chain does
not: your prompt declares `{text}`, so you must supply `{"text": ...}` with a
string. That join — `"\n\n".join(d.page_content for d in docs)` — is a step the
old loader did for you invisibly.

**4. Expecting `.batch()` to parallelize one item**

`Runnable.batch` short-circuits when given a single input, so a one-chunk corpus
shows no speed-up. That is a property of your data, not a bug.

---
## 🧪 Part 10: Try it yourself

1. The map-reduce above uses the same model for both steps. Change `reduce_chain`
   to use a larger model while `map_chain` keeps the small one — a common
   cost-saving pattern that the `chain_type="map_reduce"` string could not express
   without subclassing.
2. Add a per-chunk failure guard: wrap `map_chain` with `.with_retry()` so one bad
   chunk does not sink the batch. Then explain why doing the same thing inside
   `load_summarize_chain` would have required editing LangChain's source.
3. Run the refine loop with `docs` reversed. Does the summary change? Use that to
   explain the order-sensitivity note in Part 7.

---
## 📝 Summary

### 1. The loader hid three ordinary patterns
- **Key point**: `stuff` = join + one call; `map_reduce` = `.batch()` + one more call; `refine` = a loop
- **Key point**: written out, each is a few lines and every prompt and model is yours to change

### 2. Deprecation you cannot hear
- **Key point**: `load_summarize_chain` returns a `@deprecated` `StuffDocumentsChain` and warns about nothing at construction
- **Key point**: the only warning you get is about `.run()`, and it arrives when you call, not when you build

### 3. What replaces what
- **Key point**: `chain.run(docs)` → `chain.invoke({"text": joined})` — you now do the joining
- **Key point**: `langchain_classic.chains.summarize` exists for staged ports, not as a destination

### Next Steps
- Compare against the rewritten `8.1_Text_Summarization.ipynb`, which applies all three strategies to a real PDF
- Then `3.5_Chain_Migrations.ipynb` for the same before/after treatment of `LLMChain` and `RetrievalQA`